# NERISAL: INT8 DM-Count for Arm (Google Colab)

Makes an **INT8** version of our density model (DM-Count, UCF-QNRF weights, 432×240 input) for ExecuTorch + XNNPACK,
then compares it with FP32 on the same images: count difference and speed.

Same method as our INT8 head models: post-training quantisation, per-channel weights, calibrated on real crowd photos.
The final 1×1 density layer is kept in FP32 (it outputs very small values that INT8 would round away).

**Optional but useful:** upload a few of our Indian test photos into a folder called `test_imgs` in the file panel
before running cell 4. Runtime: CPU is fine, about 15 minutes.

In [ ]:
# 1. Install (ExecuTorch pinned to the Pi's version), DM-Count code and pretrained weights
!pip install -q executorch==1.6.0 gdown huggingface_hub
!pip show executorch | grep Version
!git clone -q https://github.com/cvlab-stonybrook/DM-Count.git
!gdown -q --folder https://drive.google.com/drive/folders/10U7F4iW_aPICM5-qJq21SXLLkzlum9tX -O dm_weights
import glob, os
W = sorted(glob.glob("dm_weights/**/*.pth", recursive=True))
print("weights:", [os.path.basename(w) for w in W])

In [ ]:
# 2. Load the QNRF model and the exact pre-processing our Pi code uses
import sys, torch, numpy as np, cv2
sys.path.insert(0, "DM-Count")
from models import vgg19
path = [w for w in W if "qnrf" in os.path.basename(w).lower()][0]
dm = vgg19(); dm.load_state_dict(torch.load(path, map_location="cpu")); dm.eval()
print("loaded", os.path.basename(path))

MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)
SIZE = (432, 240)
def prep(img_bgr):
    x = cv2.resize(img_bgr, SIZE, interpolation=cv2.INTER_AREA)[:, :, ::-1]
    x = (x.astype(np.float32) / 255 - MEAN) / STD
    return torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1))).unsqueeze(0)

class DensityOnly(torch.nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        return self.m(x)[0]                # density map; sum = count
model = DensityOnly(dm).eval()
for n, mod in model.named_modules():
    if n.count(".") <= 1: print(" ", n, type(mod).__name__)

In [ ]:
# 3. Calibration photos: 220 CrowdHuman crowd photos, downloaded automatically
import zipfile, random
os.makedirs("calib", exist_ok=True)
from huggingface_hub import list_repo_files, hf_hub_download
REPO = "ShantyCam/crowdhuman"
f = [x for x in list_repo_files(REPO, repo_type="dataset") if x.endswith("CrowdHuman_val.zip")][0]
zp = hf_hub_download(REPO, f, repo_type="dataset", local_dir="ch")
with zipfile.ZipFile(zp) as zf:
    jpgs = sorted(n for n in zf.namelist() if n.lower().endswith(".jpg"))
    for n in random.Random(1).sample(jpgs, 220):
        zf.extract(n, "calib")
os.remove(zp)
imgs = sorted(glob.glob("calib/**/*.jpg", recursive=True)); random.Random(1).shuffle(imgs)
test_imgs, calib_imgs = imgs[:20], imgs[20:140]
print(len(calib_imgs), "for calibration,", len(test_imgs), "CrowdHuman test photos")

In [ ]:
# 4. Extra test photos: DM-Count's examples + anything you uploaded to test_imgs/ (e.g. our Indian photos)
os.makedirs("test_imgs", exist_ok=True)
extra = sorted(glob.glob("DM-Count/example_images/*") + glob.glob("test_imgs/*"))
extra = [p for p in extra if cv2.imread(p) is not None]
print(len(extra), "extra test photos:", [os.path.basename(p) for p in extra])

def views(img):
    """What the Pi feeds DM-Count: the whole frame, and the 4 tiles of a 2x2 grid."""
    h, w = img.shape[:2]; out = [img]
    for i in range(2):
        for j in range(2):
            out.append(img[i * h // 2:(i + 1) * h // 2, j * w // 2:(j + 1) * w // 2])
    return out

In [ ]:
# 5. Export FP32 and INT8 (.pte); INT8 keeps the final density layer in FP32
import time
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config
try:
    from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
except ImportError:
    from torch.ao.quantization.quantize_pt2e import prepare_pt2e, convert_pt2e

x0 = torch.zeros(1, 3, SIZE[1], SIZE[0])
os.makedirs("pte", exist_ok=True)
with torch.no_grad():
    ep = torch.export.export(model, (x0,))
open("pte/dmcount_qnrf_432x240_fp32.pte", "wb").write(
    to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()]).to_executorch().buffer)

q = XNNPACKQuantizer().set_global(get_symmetric_quantization_config(is_per_channel=True))
kept = None
for name in ("m.density_layer", "density_layer"):          # keep the last layer FP32
    try:
        q.set_module_name(name, None); kept = name; break
    except Exception as e:
        print("could not exclude", name, e)
print("final density layer kept FP32:" if kept else "WARNING: whole model INT8", kept or "")
gm = torch.export.export(model, (x0,)).module()
gm = prepare_pt2e(gm, q)
t = time.time()
with torch.no_grad():
    for p in calib_imgs:
        im = cv2.imread(p)
        if im is None: continue
        for v in views(im)[:3]:                            # whole frame + two tiles: realistic value ranges
            gm(prep(v))
gm = convert_pt2e(gm)
print(f"calibrated in {time.time() - t:.0f} s")
ep8 = torch.export.export(gm, (x0,))
open("pte/dmcount_qnrf_432x240_int8.pte", "wb").write(
    to_edge_transform_and_lower(ep8, partitioner=[XnnpackPartitioner()]).to_executorch().buffer)
for f in sorted(glob.glob("pte/*.pte")): print(f, f"{os.path.getsize(f)/1e6:.1f} MB")

In [ ]:
# 6. FP32 vs INT8: counts on the same photos (whole frame + 2x2 tiles, like the Pi) and Colab CPU time
from executorch.runtime import Runtime
M = {k: Runtime.get().load_program(f"pte/dmcount_qnrf_432x240_{k}.pte").load_method("forward") for k in ("fp32", "int8")}
def count(meth, img):
    return sum(float(meth.execute([prep(v)])[0].sum()) for v in views(img)[1:])   # 2x2 tiled count
rows, diffs, T = [], [], {"fp32": [], "int8": []}
for p in test_imgs + extra:
    im = cv2.imread(p)
    if im is None: continue
    c = {}
    for k, m in M.items():
        t = time.perf_counter(); c[k] = count(m, im); T[k].append((time.perf_counter() - t) * 1000 / 4)
    rows.append((os.path.basename(p)[:22], c["fp32"], c["int8"]))
    if c["fp32"] >= 5: diffs.append(abs(c["int8"] - c["fp32"]) / c["fp32"] * 100)
print(f"{'photo':24s}{'FP32':>9s}{'INT8':>9s}")
for r in rows: print(f"{r[0]:24s}{r[1]:9.1f}{r[2]:9.1f}")
print(f"\nmean count difference (photos with 5+ people): {np.mean(diffs):.1f}%   median {np.median(diffs):.1f}%")
print(f"Colab CPU per tile: FP32 {np.mean(T['fp32']):.0f} ms, INT8 {np.mean(T['int8']):.0f} ms (the Pi decides the real speed)")

In [ ]:
# 7. Download the INT8 model (copy dmcount_qnrf_432x240_int8.pte to ~/nerisal on the Pi)
from google.colab import files
files.download("pte/dmcount_qnrf_432x240_int8.pte")